<a href="https://colab.research.google.com/github/EnzoSalvini/Calculo_numerico/blob/main/AtividadeAnimacaoGauss.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter

# ---------------------------------------------------------------
# Dados alteráveis: NECESSARIO matriz 3x3
# ---------------------------------------------------------------
A_g = np.array([[2., 1., -1.], [-3., -1., 2.], [-2., 1., 2.]])
b_g = np.array([8., -11., -3.])
M0 = np.column_stack((A_g, b_g))


# ---------------------------------------------------------------
# 1) Coleta cada operação L_i <- L_i - m*L_k
# ---------------------------------------------------------------
def coletar_passos(M):
    M = M.copy()
    n = len(M)
    passos = []
    for k in range(n - 1):
        for i in range(k + 1, n):
            m = M[i, k] / M[k, k]
            antes = M.copy()
            M[i, :] -= m * M[k, :]
            passos.append(dict(k=k, i=i, m=m, antes=antes, depois=M.copy()))
    return passos


passos = coletar_passos(M0)
x_sol = np.linalg.solve(A_g, b_g)

# Substituição regressiva a partir da matriz triangular final
U = passos[-1]["depois"]


def fmt(v):
    s = f"{v:.2f}"
    return "0.00" if s in ("-0.00", "0.00") else s


z = U[2, 3] / U[2, 2]
y = (U[1, 3] - U[1, 2] * z) / U[1, 1]
x = (U[0, 3] - U[0, 1] * y - U[0, 2] * z) / U[0, 0]

EQS = [
    f"z = {fmt(U[2,3])} / {fmt(U[2,2])} = {fmt(z)}",
    f"y = ({fmt(U[1,3])} − ({fmt(U[1,2])})·({fmt(z)})) / {fmt(U[1,1])} = {fmt(y)}",
    f"x = ({fmt(U[0,3])} − ({fmt(U[0,1])})·({fmt(y)}) − ({fmt(U[0,2])})·({fmt(z)})) / {fmt(U[0,0])} = {fmt(x)}",
]

# ---------------------------------------------------------------
# 2) Linha do tempo: quantos frames cada fase dura
# ---------------------------------------------------------------
N_INI, N_DEST, N_MULT, N_SUB, N_HOLD, N_RET, N_FIM = 20, 15, 20, 30, 15, 35, 60

timeline = [("ini", None, 0)] * N_INI
for p in range(len(passos)):
    timeline += [("dest", p, f) for f in range(N_DEST)]
    timeline += [("mult", p, f) for f in range(N_MULT)]
    timeline += [("sub", p, f) for f in range(N_SUB)]
    timeline += [("hold", p, f) for f in range(N_HOLD)]
timeline += [("retro", r, f) for r in range(3) for f in range(N_RET)]
timeline += [("fim", None, f) for f in range(N_FIM)]

# ---------------------------------------------------------------
# 3) Desenho
# ---------------------------------------------------------------
BG, FG = "#0f172a", "#e2e8f0"
C_PIVO, C_ALVO, C_MULT, C_OK = "#fbbf24", "#38bdf8", "#f472b6", "#4ade80"
XS = [2.2, 3.5, 4.8, 6.6]          # posição x das 4 colunas
YS = [4.4, 3.6, 2.8]               # posição y das 3 linhas
Y_MULT = 1.7

fig = plt.figure(figsize=(15, 6.5), facecolor=BG)
ax = fig.add_axes([0.0, 0.0, 0.55, 1.0])                        # matriz
ax3 = fig.add_axes([0.52, 0.04, 0.47, 0.92], projection="3d")   # planos


def ease(t):                        # suaviza o movimento (ease in-out)
    return t * t * (3 - 2 * t)


def colchetes():
    x0, x1, y0, y1 = 1.3, 7.5, 2.35, 4.85
    for xa, xb in ((x0, x0 + 0.15), (x1, x1 - 0.15)):
        ax.plot([xb, xa, xa, xb], [y1, y1, y0, y0], color=FG, lw=2)
    ax.plot([5.7, 5.7], [y0 + 0.05, y1 - 0.05], color=FG, lw=1.2, ls="--", alpha=.6)


def desenhar_linha(vals, y, cor, alpha=1.0, peso="normal"):
    for x, v in zip(XS, vals):
        ax.text(x, y, fmt(v), ha="center", va="center", fontsize=20,
                color=cor, alpha=alpha, fontweight=peso, family="monospace")


LIMS = [(-2, 6), (-1, 7), (-5, 3)]                 # janela de x, y, z
COR_PLANO = ["#f87171", "#60a5fa", "#c084fc"]      # L1, L2, L3


def desenhar3d(mat, alvo, idx, fase, f):
    """Desenha os 3 planos a1·x + a2·y + a3·z = b (uma linha da matriz cada)."""
    ax3.cla()
    ax3.set_facecolor(BG)
    ax3.set_xlim(*LIMS[0]); ax3.set_ylim(*LIMS[1]); ax3.set_zlim(*LIMS[2])
    ax3.set_box_aspect((1, 1, 1))
    ax3.tick_params(colors=FG, labelsize=8)
    for nome, eixo in zip("xyz", (ax3.xaxis, ax3.yaxis, ax3.zaxis)):
        eixo.set_pane_color((1, 1, 1, 0.03))
        eixo._axinfo["grid"]["color"] = (1, 1, 1, 0.15)
        eixo.label.set_color(FG)
    ax3.set_xlabel("x"); ax3.set_ylabel("y"); ax3.set_zlabel("z")

    for r in range(3):
        coef = np.array(mat[r][:3], dtype=float)
        if np.linalg.norm(coef) < 1e-9:
            continue
        dom = int(np.argmax(np.abs(coef)))          # variável isolada (mais estável)
        o1, o2 = [j for j in range(3) if j != dom]
        g1, g2 = np.meshgrid(np.linspace(*LIMS[o1], 16), np.linspace(*LIMS[o2], 16))
        W = (mat[r][3] - coef[o1] * g1 - coef[o2] * g2) / coef[dom]
        W = np.where((W >= LIMS[dom][0]) & (W <= LIMS[dom][1]), W, np.nan)
        pts = [None, None, None]
        pts[o1], pts[o2], pts[dom] = g1, g2, W
        alpha = 0.55 if r == alvo else 0.28
        if fase == "fim":
            alpha = 0.28 - 0.12 * ease(min(1.0, f / 20))   # deixa o ponto visível
        ax3.plot_surface(*pts, color=COR_PLANO[r], alpha=alpha,
                         linewidth=0, antialiased=False, shade=False)

    # ponto da solução aparece no final
    if fase == "fim":
        a = ease(min(1.0, f / 20))
        ax3.scatter([x], [y], [z], s=320 * a, color=C_OK, edgecolors="white",
                    linewidths=1.5, depthshade=False, alpha=a)
        ax3.text(x + 0.3, y, z + 0.8, f"({fmt(x)}, {fmt(y)}, {fmt(z)})",
                 color=C_OK, fontsize=13, fontweight="bold", alpha=a)

    # legenda e título do painel
    ax3.text2D(0.5, 0.97, "Interseção dos planos", transform=ax3.transAxes,
               ha="center", color=FG, fontsize=15, fontweight="bold")
    for r in range(3):
        ax3.text2D(0.33 + 0.13 * r, 0.925, f"■ L{r+1}", transform=ax3.transAxes,
                   color=COR_PLANO[r], fontsize=12, fontweight="bold")

    ax3.view_init(elev=20, azim=35 + idx * 0.3)     # câmera girando devagar



def desenhar(idx):
    ax.clear()
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 6)
    ax.axis("off")
    ax.set_facecolor(BG)

    fase, p, f = timeline[idx]

    if fase == "ini":
        mat, titulo, info = M0, "Matriz aumentada [A | b]", ""
        k = i = None
    elif fase in ("retro", "fim"):
        mat = U
        k = i = None
        if fase == "retro":
            titulo = "Substituição regressiva (de baixo para cima)"
            info = ""
        else:
            titulo = "Solução do sistema"
            info = f"x = {fmt(x)}     y = {fmt(y)}     z = {fmt(z)}"
    else:
        s = passos[p]
        k, i, m = s["k"], s["i"], s["m"]
        mat = s["antes"].copy()
        titulo = f"Passo {p+1}/{len(passos)}:  L{i+1} ← L{i+1} − m{i+1}{k+1}·L{k+1}"
        num, den = s["antes"][i, k], s["antes"][k, k]
        info = f"m{i+1}{k+1} = {fmt(num)} / {fmt(den)} = {fmt(m)}"

        prod = m * s["antes"][k]
        if fase == "sub":
            t = ease(f / (N_SUB - 1))
            mat[i] = s["antes"][i] + t * (s["depois"][i] - s["antes"][i])
        elif fase == "hold":
            mat = s["depois"].copy()

    # título e informação
    ax.text(5, 5.6, titulo, ha="center", fontsize=17, color=FG, fontweight="bold")
    ax.text(5, 0.2, info, ha="center", fontsize=20 if fase == "fim" else 16,
            color=C_OK if fase == "fim" else C_MULT, family="monospace",
            fontweight="bold")

    colchetes()

    # matriz
    for r in range(3):
        cor, peso = FG, "normal"
        if k is not None and r == k:
            cor, peso = C_PIVO, "bold"
        if i is not None and r == i:
            cor, peso = C_ALVO, "bold"
        ax.text(0.9, YS[r], f"L{r+1}", ha="right", va="center", fontsize=14,
                color=cor, alpha=.8, family="monospace")
        desenhar_linha(mat[r], YS[r], cor, peso=peso)

    # linha "m · L_pivô"
    if fase in ("mult", "sub"):
        a = ease(f / (N_MULT - 1)) if fase == "mult" else 1.0
        if fase == "sub":
            a = 1 - 0.6 * ease(f / (N_SUB - 1))   # esmaece enquanto é subtraída
        ax.plot([1.6, 7.3], [2.15, 2.15], color=FG, lw=1, alpha=.4)
        ax.text(0.9, Y_MULT, f"{fmt(m)}·L{k+1}", ha="right", va="center",
                fontsize=13, color=C_MULT, alpha=a, family="monospace")
        desenhar_linha(prod, Y_MULT, C_MULT, alpha=a)
        ax.text(7.9, Y_MULT, "(subtrair)", va="center", fontsize=12,
                color=C_MULT, alpha=a * .8)

    # equações da substituição regressiva
    if fase in ("retro", "fim"):
        for r_idx, txt in enumerate(EQS):
            if fase == "retro":
                if r_idx > p:
                    continue
                a = ease(f / (N_RET - 1)) if r_idx == p else 1.0
            else:
                a = 1.0
            ax.text(5, 1.85 - 0.55 * r_idx, txt, ha="center", va="center",
                    fontsize=14, color=C_OK if fase == "fim" else FG,
                    alpha=a, family="monospace")

    # seta ligando o pivô ao alvo durante o destaque
    if fase == "dest":
        a = ease(f / (N_DEST - 1))
        ax.annotate("", xy=(8.3, YS[i]), xytext=(8.3, YS[k]),
                    arrowprops=dict(arrowstyle="->", color=C_MULT, lw=2, alpha=a))

    desenhar3d(mat, i, idx, fase, f)


# ---------------------------------------------------------------
# 4) Rodar
# ---------------------------------------------------------------
anim = FuncAnimation(fig, desenhar, frames=len(timeline), interval=40, repeat=False)

if __name__ == "__main__":
    # salva como GIF (funciona em qualquer lugar)
    anim.save("gauss_animado.gif", writer=PillowWriter(fps=25), dpi=100)
    print("Salvo em gauss_animado.gif")

    print("\nSolução do sistema:")
    print(f"x = {x:.4f}")
    print(f"y = {y:.4f}")
    print(f"z = {z:.4f}")
    print("Conferência (np.linalg.solve):", x_sol)

# No Jupyter / Colab, em vez de salvar, use:
#   from IPython.display import HTML
#   HTML(anim.to_jshtml())

Salvo em gauss_animado.gif

Solução do sistema:
x = 2.0000
y = 3.0000
z = -1.0000
Conferência (np.linalg.solve): [ 2.  3. -1.]
